In [1]:
!pip install -q mediapipe==0.10.35

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 14.2 MB/s eta 0:00:00


In [2]:
import os, subprocess, json
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists("/content/raw"):
    subprocess.run(["unzip", "-q", "-o", "/content/drive/MyDrive/raw_videos.zip", "-d", "/content/"], check=True)

with open("/content/drive/MyDrive/metadata.json") as f:
    metadata = json.load(f)
with open("/content/drive/MyDrive/split_v1.json") as f:
    split = json.load(f)

def vid_id(name): return name.replace(".mp4", "")
label_of = {vid_id(n): (0 if i["label"] == "REAL" else 1) for n, i in metadata.items()}

print("Videos found:", len([f for f in os.listdir("/content/raw") if f.endswith(".mp4")]))
print("Split counts:", {k: len(v) for k, v in split.items()})

Mounted at /content/drive
Videos found: 401
Split counts: {'train': 292, 'val': 54, 'test': 54}


In [3]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
import urllib.request
import cv2
import numpy as np

model_path = "/content/face_landmarker.task"
if not os.path.exists(model_path):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task",
        model_path
    )

base_options = mp_python.BaseOptions(model_asset_path=model_path)
options = mp_vision.FaceLandmarkerOptions(base_options=base_options, running_mode=mp_vision.RunningMode.IMAGE, num_faces=1)
landmarker = mp_vision.FaceLandmarker.create_from_options(options)
print("Landmarker ready")

Landmarker ready


In [7]:
detector_model_path = "/content/face_detector.tflite"
if not os.path.exists(detector_model_path):
    try:
        urllib.request.urlretrieve(
            "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_full_range/float16/1/blaze_face_full_range.tflite",
            detector_model_path
        )
        print("Downloaded FULL-range detector")
    except Exception as e:
        print("Using short-range fallback:", e)
        urllib.request.urlretrieve(
            "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite",
            detector_model_path
        )

det_base_options = mp_python.BaseOptions(model_asset_path=detector_model_path)
det_options = mp_vision.FaceDetectorOptions(base_options=det_base_options, min_detection_confidence=0.3)
face_detector = mp_vision.FaceDetector.create_from_options(det_options)

def detect_and_crop(rgb, margin=0.35):
    h, w, _ = rgb.shape
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))
    result = face_detector.detect(mp_img)
    if not result.detections:
        return None
    box = result.detections[0].bounding_box
    x1 = max(box.origin_x - int(box.width * margin), 0)
    y1 = max(box.origin_y - int(box.height * margin), 0)
    x2 = min(box.origin_x + box.width + int(box.width * margin), w)
    y2 = min(box.origin_y + box.height + int(box.height * margin), h)
    crop = rgb[y1:y2, x1:x2]
    return np.ascontiguousarray(crop) if crop.size > 0 else None

print("Detector ready")

Downloaded FULL-range detector
Detector ready


In [8]:
T = 80

def sample_frame_indices(total_frames, T=80, start_frac=0.05, end_frac=0.95):
    start = int(total_frames * start_frac)
    end = int(total_frames * end_frac)
    if end <= start:
        start, end = 0, total_frames
    if end - start < T:
        idx = list(range(start, end))
    else:
        idx = list(np.linspace(start, end - 1, T).astype(int))
    seen, deduped = set(), []
    for x in idx:
        if x not in seen:
            deduped.append(x)
            seen.add(x)
    idx = deduped
    while len(idx) < T:
        idx.append(idx[-1] if idx else 0)
    return idx[:T]

def normalize_landmarks(lm):
    center = lm.mean(axis=0)
    centered = lm - center
    scale = np.sqrt((centered ** 2).sum(axis=1)).mean()
    scale = max(scale, 1e-6)
    return (centered / scale).astype(np.float32)

print("Sampling spans full video now, T =", T)

Sampling spans full video now, T = 80


In [9]:
def extract_sequence(video_path):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 2:
        cap.release()
        return None
    wanted = set(sample_frame_indices(total, T))
    frames = {}
    i = 0
    while cap.isOpened() and len(frames) < len(wanted):
        ret, frame = cap.read()
        if not ret:
            break
        if i in wanted:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            crop = detect_and_crop(rgb)
            if crop is not None:
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=crop)
                result = landmarker.detect(mp_image)
                if result.face_landmarks:
                    pts = np.array([[p.x, p.y] for p in result.face_landmarks[0]])
                    frames[i] = normalize_landmarks(pts)
        i += 1
    cap.release()
    if not frames:
        return None
    idx_order = sample_frame_indices(total, T)
    last_valid = None
    seq = []
    for fi in idx_order:
        if fi in frames:
            last_valid = frames[fi]
        if last_valid is None:
            return None
        seq.append(last_valid)
    return np.stack(seq).reshape(T, -1)

for fname in list(label_of.keys())[:3]:
    fpath = f"/content/raw/{fname}.mp4"
    if os.path.exists(fpath):
        seq = extract_sequence(fpath)
        print(fname, "->", None if seq is None else seq.shape)

aagfhgtpmv -> (80, 956)
aapnvogymq -> (80, 956)
abarnvbtwb -> (80, 956)


In [11]:
sequences = {}
labels = {}
failed = []

video_files = [f for f in os.listdir("/content/raw") if f.endswith(".mp4")]

for idx, fname in enumerate(video_files):
    vid = vid_id(fname)
    if vid not in label_of:
        continue
    seq = extract_sequence(os.path.join("/content/raw", fname))
    if seq is None:
        failed.append(fname)
        continue
    sequences[vid] = seq
    labels[vid] = label_of[vid]
    if idx % 50 == 0:
        print(f"Processed {idx}/{len(video_files)}")

print(f"\nExtracted: {len(sequences)} | Failed: {len(failed)}")
if failed:
    print("Failed videos:", failed[:15])

np.savez_compressed(
    "/content/drive/MyDrive/facial_sequences_v2.npz",
    **{f"X_{v}": s for v, s in sequences.items()},
    **{f"y_{v}": np.array(labels[v]) for v in sequences}
)
print("Saved facial_sequences_v2.npz to Drive")

Processed 0/401
Processed 50/401
Processed 100/401
Processed 150/401
Processed 200/401
Processed 250/401
Processed 300/401
Processed 350/401
Processed 400/401

Extracted: 389 | Failed: 11
Failed videos: ['cmxcfkrjiv.mp4', 'cwqlvzefpg.mp4', 'djvutyvaio.mp4', 'atvmxvwyns.mp4', 'dqppxmoqdl.mp4', 'dxbqjxrhin.mp4', 'dlrsbscitn.mp4', 'andaxzscny.mp4', 'bgmlwsoamc.mp4', 'cffffbcywc.mp4', 'crzfebnfgb.mp4']
Saved facial_sequences_v2.npz to Drive


In [12]:
data = np.load("/content/drive/MyDrive/facial_sequences_v2.npz")
all_vids = sorted(set(k[2:] for k in data.files if k.startswith("X_")))

def build_split(video_list):
    X, y, vids = [], [], []
    for v in video_list:
        if v in all_vids:
            X.append(data[f"X_{v}"])
            y.append(int(data[f"y_{v}"]))
            vids.append(v)
    return np.array(X, dtype=np.float32), np.array(y), vids

X_train, y_train, vids_train = build_split(split["train"])
X_val, y_val, vids_val = build_split(split["val"])
X_test, y_test, vids_test = build_split(split["test"])
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 281 | Val: 54 | Test: 54


In [16]:
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

def eye_aspect_ratio(pts, eye_idx):
    p = pts[eye_idx]
    vert1 = np.linalg.norm(p[1] - p[5])
    vert2 = np.linalg.norm(p[2] - p[4])
    horiz = np.linalg.norm(p[0] - p[3])
    return (vert1 + vert2) / (2.0 * horiz + 1e-6)

def mouth_aspect_ratio(pts):
    upper, lower = pts[13], pts[14]
    left, right = pts[61], pts[291]
    vert = np.linalg.norm(upper - lower)
    horiz = np.linalg.norm(left - right)
    return vert / (horiz + 1e-6)

def count_threshold_dips(series, drop_frac=0.25):
    series = np.asarray(series)
    if series.std() < 1e-8:
        return 0
    thresh = series.mean() * (1 - drop_frac)
    below = (series < thresh).astype(int)
    return int((np.diff(below) == 1).sum())

def count_threshold_rises(series, rise_frac=0.25):
    series = np.asarray(series)
    if series.std() < 1e-8:
        return 0
    thresh = series.mean() * (1 + rise_frac)
    above = (series > thresh).astype(int)
    return int((np.diff(above) == 1).sum())

def compute_features(seq_flat):
    seq = seq_flat.reshape(T, 478, 2)
    ear_series = np.array([(eye_aspect_ratio(f, LEFT_EYE) + eye_aspect_ratio(f, RIGHT_EYE)) / 2 for f in seq])
    mar_series = np.array([mouth_aspect_ratio(f) for f in seq])
    motion = np.abs(np.diff(seq.reshape(T, -1), axis=0)).mean(axis=1)

    return np.array([
        ear_series.mean(), ear_series.std(), ear_series.min(), ear_series.max(),
        mar_series.mean(), mar_series.std(), mar_series.min(), mar_series.max(),
        motion.mean(), motion.std(), motion.max(),
        count_threshold_dips(ear_series),
        count_threshold_rises(mar_series),
    ], dtype=np.float32)

feature_names = ["ear_mean","ear_std","ear_min","ear_max","mar_mean","mar_std","mar_min","mar_max",
                  "motion_mean","motion_std","motion_max","blink_count","mouth_open_count"]

def build_feature_split(X, y, vids):
    feats = np.array([compute_features(x) for x in X])
    return feats, y, vids

Xf_train, yf_train, vids_train_f = build_feature_split(X_train, y_train, vids_train)
Xf_val, yf_val, vids_val_f = build_feature_split(X_val, y_val, vids_val)
Xf_test, yf_test, vids_test_f = build_feature_split(X_test, y_test, vids_test)

print("Feature count:", len(feature_names))
print(f"Train: {len(Xf_train)} | Val: {len(Xf_val)} | Test: {len(Xf_test)}")
print("\nblink_count (first 10):", Xf_train[:10, 11])
print("mouth_open_count (first 10):", Xf_train[:10, 12])
print("mar_mean (first 10):", Xf_train[:10, 4])

Feature count: 13
Train: 281 | Val: 54 | Test: 54

blink_count (first 10): [ 2.  4.  6.  6.  3.  4. 10. 10.  2.  2.]
mouth_open_count (first 10): [14. 13. 13. 14. 13. 13.  7. 11. 18. 16.]
mar_mean (first 10): [0.19540925 0.22958979 0.15140507 0.20269473 0.15704587 0.10335632
 0.25345597 0.19509722 0.20491557 0.17686825]


In [17]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, confusion_matrix
import pandas as pd

scaler = StandardScaler()
Xf_train_s = scaler.fit_transform(Xf_train)
Xf_val_s = scaler.transform(Xf_val)
Xf_test_s = scaler.transform(Xf_test)

w_real = len(yf_train) / (2 * (yf_train == 0).sum())
w_fake = len(yf_train) / (2 * (yf_train == 1).sum())
class_weight = {0: w_real, 1: w_fake}
print("Class weights:", class_weight)

clf = LogisticRegressionCV(
    Cs=10, cv=5, class_weight=class_weight, max_iter=2000,
    scoring="balanced_accuracy", random_state=42
)
clf.fit(Xf_train_s, yf_train)
print("Chosen regularization strength C:", clf.C_[0])

def summarize(probs, labels, title):
    preds = (probs >= 0.5).astype(int)
    print(title)
    print(f"  Balanced accuracy: {balanced_accuracy_score(labels, preds)*100:.2f}%")
    print(f"  Real correct: {(preds[labels==0]==0).mean()*100:.2f}% | Fake correct: {(preds[labels==1]==1).mean()*100:.2f}%")
    print(f"  AUC: {roc_auc_score(labels, probs):.4f}")
    print(f"  Confusion matrix:\n{confusion_matrix(labels, preds)}")

val_probs = clf.predict_proba(Xf_val_s)[:, 1]
summarize(val_probs, yf_val, "VALIDATION")

Class weights: {0: np.float64(2.7549019607843137), 1: np.float64(0.6108695652173913)}
Chosen regularization strength C: 21.54434690031882
VALIDATION
  Balanced accuracy: 67.26%
  Real correct: 75.00% | Fake correct: 59.52%
  AUC: 0.7361
  Confusion matrix:
[[ 9  3]
 [17 25]]


In [18]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

X_combined = np.vstack([Xf_train, Xf_val])
y_combined = np.concatenate([yf_train, yf_val])
X_combined_s = StandardScaler().fit_transform(X_combined)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    LogisticRegressionCV(Cs=10, cv=5, class_weight="balanced", max_iter=2000, random_state=42),
    X_combined_s, y_combined, cv=cv, scoring="balanced_accuracy"
)
print(f"5-fold CV balanced accuracy: {cv_scores.mean()*100:.2f}% (+/- {cv_scores.std()*100:.2f}%)")
print("Individual fold scores:", [f"{s*100:.1f}%" for s in cv_scores])

5-fold CV balanced accuracy: 59.46% (+/- 7.82%)
Individual fold scores: ['52.0%', '53.0%', '60.4%', '58.1%', '73.8%']


In [19]:
test_probs = clf.predict_proba(Xf_test_s)[:, 1]
summarize(test_probs, yf_test, "TEST")

TEST
  Balanced accuracy: 66.07%
  Real correct: 75.00% | Fake correct: 57.14%
  AUC: 0.6111
  Confusion matrix:
[[ 9  3]
 [18 24]]


In [20]:
import pickle

with open("/content/drive/MyDrive/facial_model.pkl", "wb") as f:
    pickle.dump({"model": clf, "scaler": scaler, "feature_names": feature_names}, f)

val_df = pd.DataFrame({"video": vids_val_f, "fake_prob": val_probs, "label": yf_val})
test_df = pd.DataFrame({"video": vids_test_f, "fake_prob": test_probs, "label": yf_test})
val_df.to_csv("/content/drive/MyDrive/facial_val_video_scores.csv", index=False)
test_df.to_csv("/content/drive/MyDrive/facial_test_video_scores.csv", index=False)

info = {
    "model": "LogisticRegressionCV on hand-crafted facial motion features (EAR, MAR via landmarks 13/14/61/291, motion, blink/mouth-open counts)",
    "sampling": "80 frames spread across 5%-95% of full video",
    "features": feature_names,
    "split_file": "split_v1.json",
    "chosen_C": float(clf.C_[0]),
    "class_weights": {"real": float(w_real), "fake": float(w_fake)},
    "seed": 42,
    "val_balanced_acc": float(balanced_accuracy_score(yf_val, (val_probs>=0.5).astype(int))),
    "val_auc": float(roc_auc_score(yf_val, val_probs)),
    "test_balanced_acc": float(balanced_accuracy_score(yf_test, (test_probs>=0.5).astype(int))),
    "test_auc": float(roc_auc_score(yf_test, test_probs)),
    "cv_balanced_acc_mean": float(cv_scores.mean()),
    "cv_balanced_acc_std": float(cv_scores.std()),
}
with open("/content/drive/MyDrive/facial_model_info.json", "w") as f:
    json.dump(info, f, indent=2)
print(json.dumps(info, indent=2))

{
  "model": "LogisticRegressionCV on hand-crafted facial motion features (EAR, MAR via landmarks 13/14/61/291, motion, blink/mouth-open counts)",
  "sampling": "80 frames spread across 5%-95% of full video",
  "features": [
    "ear_mean",
    "ear_std",
    "ear_min",
    "ear_max",
    "mar_mean",
    "mar_std",
    "mar_min",
    "mar_max",
    "motion_mean",
    "motion_std",
    "motion_max",
    "blink_count",
    "mouth_open_count"
  ],
  "split_file": "split_v1.json",
  "chosen_C": 21.54434690031882,
  "class_weights": {
    "real": 2.7549019607843137,
    "fake": 0.6108695652173913
  },
  "seed": 42,
  "val_balanced_acc": 0.6726190476190477,
  "val_auc": 0.736111111111111,
  "test_balanced_acc": 0.6607142857142857,
  "test_auc": 0.6111111111111112,
  "cv_balanced_acc_mean": 0.5946153846153845,
  "cv_balanced_acc_std": 0.07815994210062717
}
